# Appendix A1 — Pydantic from Zero

**Who this is for:** a complete beginner. By the end you will understand *every* Pydantic idea
this series relies on — models, fields, validation, coercion, nesting, validators, computed
fields, serialization, parsing, configuration, and settings.

**How to read it:** each idea is explained in words first, then a small **runnable code cell**
shows exactly how it behaves. Run the cells top to bottom.

> We use **Pydantic v2** (the current major version). A few method names differ from the older
> v1 (`model_dump` not `dict`, `model_validate` not `parse_obj`, `field_validator` not
> `validator`); this appendix uses v2 throughout.

In [1]:
# === Chapter A1 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter A1 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter A1 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 1. What is Pydantic, and why does it exist?

Python's type hints (`name: str`, `age: int`) are normally **just documentation** — Python does
not enforce them at runtime. You can happily write `age = "hello"` and nothing complains until
something breaks far away.

**Pydantic** turns those type hints into *runtime rules*. You describe the shape of your data as a
class, and Pydantic **validates** incoming data against it, **coerces** compatible types, and
gives **clear errors** when data is wrong. It solves three everyday problems:

1. **Validation** — reject bad data at the boundary (an API request, a JSON file, an LLM output).
2. **Parsing** — turn loose input (JSON, dicts, strings) into typed Python objects.
3. **Serialization** — turn those objects back into dicts/JSON.

That is exactly why this series models a patent as a Pydantic `PatentDocument` instead of a bag of
dicts: the structure is *guaranteed*, not hoped for.

In [2]:
# Plain Python type hints are NOT enforced — this runs without complaint:
def make_person(name: str, age: int):
    return {"name": name, "age": age}

bad = make_person("Ada", "not a number")   # age should be int... Python does not care
print("plain python accepts nonsense:", bad)

plain python accepts nonsense: {'name': 'Ada', 'age': 'not a number'}


## 2. Your first model

A **model** is a class that inherits from `BaseModel`. Each **field** is a class attribute with a
type hint. Creating an instance validates the data and gives you a normal object with typed
attributes.

In [3]:
from pydantic import BaseModel

class Person(BaseModel):
    name: str          # a required string field
    age: int           # a required integer field

p = Person(name="Ada", age=36)
print(p)                     # readable repr
print("attribute access:", p.name, "|", p.age, "| type of age:", type(p.age).__name__)

name='Ada' age=36
attribute access: Ada | 36 | type of age: int


## 3. Validation and error messages

If the data cannot be made to fit the types, Pydantic raises a **`ValidationError`** that says
*which field* failed and *why*. This is the whole point — problems surface immediately, at the
boundary, with a precise message.

In [4]:
from pydantic import ValidationError

try:
    Person(name="Ada", age="hello")     # "hello" is not an int and cannot be coerced
except ValidationError as e:
    print(e)                            # human-readable
    print("\nstructured errors:", e.errors()[0]["type"], "on field", e.errors()[0]["loc"])

1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='hello', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

structured errors: int_parsing on field ('age',)


## 4. Type coercion (lax vs strict)

By default Pydantic is **lax**: if a value can be *sensibly* converted to the declared type, it
will be (e.g. the string `"36"` → the int `36`). This is great for parsing web/JSON input where
everything arrives as strings. When you need exactness, you can opt into **strict** mode.

In [5]:
p = Person(name="Ada", age="36")        # "36" (str) is coerced to 36 (int)
print("coerced:", p.age, type(p.age).__name__)

from pydantic import StrictInt
class StrictPerson(BaseModel):
    age: StrictInt                      # now a string is NOT accepted
try:
    StrictPerson(age="36")
except ValidationError as e:
    print("strict rejects the string:", e.errors()[0]["type"])

coerced: 36 int
strict rejects the string: int_type


## 5. Optional fields, defaults, and `default_factory`

A field is **required** unless you give it a default. Use `Optional[T]` (or `T | None`) plus a
default of `None` for "may be missing". For **mutable** defaults (lists, dicts) never share one
instance across objects — use `default_factory` to build a fresh one each time.

In [6]:
from typing import Optional
from pydantic import Field

class Account(BaseModel):
    username: str                                   # required
    is_active: bool = True                          # default value
    nickname: Optional[str] = None                  # optional, may be None
    tags: list[str] = Field(default_factory=list)   # fresh [] per instance (safe mutable default)

a = Account(username="ada")
b = Account(username="alan")
a.tags.append("admin")
print("a:", a.model_dump())
print("b:", b.model_dump(), "  <- b.tags is independent, not shared with a")

a: {'username': 'ada', 'is_active': True, 'nickname': None, 'tags': ['admin']}
b: {'username': 'alan', 'is_active': True, 'nickname': None, 'tags': []}   <- b.tags is independent, not shared with a


## 6. `Field()` — constraints, metadata, aliases

`Field()` lets you attach **constraints** and **metadata** to a field:

- numbers: `gt`, `ge`, `lt`, `le` (greater/less than, or-equal)
- strings/collections: `min_length`, `max_length`, `pattern` (regex)
- documentation: `description`
- input renaming: `alias` (the *incoming* key name differs from the *attribute* name)

Constraints are validated automatically; violations raise `ValidationError`.

In [7]:
class Product(BaseModel):
    sku: str = Field(pattern=r"^[A-Z]{3}-\d{4}$", description="e.g. ABC-1234")
    price: float = Field(gt=0, description="must be positive")
    quantity: int = Field(ge=0, le=1000)

ok = Product(sku="ABC-1234", price=9.99, quantity=5)
print("valid:", ok.model_dump())

try:
    Product(sku="bad", price=-1, quantity=99999)   # three violations at once
except ValidationError as e:
    print("\nviolations:", [(err["loc"][0], err["type"]) for err in e.errors()])

valid: {'sku': 'ABC-1234', 'price': 9.99, 'quantity': 5}

violations: [('sku', 'string_pattern_mismatch'), ('price', 'greater_than'), ('quantity', 'less_than_equal')]


## 7. Nested models

A field's type can be **another model**. Pydantic validates the whole tree, so a nested dict of
input is turned into nested typed objects. This is how real structures (a patent → its claims →
their anchors) are represented.

In [8]:
class Address(BaseModel):
    city: str
    country: str

class Company(BaseModel):
    name: str
    headquarters: Address          # a nested model

# You can pass a dict for the nested field — Pydantic builds the Address for you:
c = Company(name="Acme", headquarters={"city": "Boston", "country": "US"})
print(type(c.headquarters).__name__, "->", c.headquarters.city)
print("whole tree:", c.model_dump())

Address -> Boston
whole tree: {'name': 'Acme', 'headquarters': {'city': 'Boston', 'country': 'US'}}


## 8. Collections of models, enums, and `Literal`

Fields can be **lists/dicts of models**. For a fixed set of allowed values use an **`Enum`** (a
named, reusable set) or **`Literal`** (an inline set of exact values). Both restrict input to the
allowed choices.

In [9]:
from enum import Enum
from typing import Literal

class Role(str, Enum):
    GUEST = "guest"
    ADMIN = "admin"

class Team(BaseModel):
    members: list[Person]                     # a list of nested models
    lead_role: Role                           # must be one of the enum values
    tier: Literal["free", "pro"]              # must be exactly one of these strings

t = Team(members=[{"name": "Ada", "age": 36}, {"name": "Alan", "age": 41}],
         lead_role="admin", tier="pro")
print("members:", [m.name for m in t.members], "| role:", t.lead_role.value, "| tier:", t.tier)

try:
    Team(members=[], lead_role="admin", tier="enterprise")   # 'enterprise' not allowed
except ValidationError as e:
    print("bad tier ->", e.errors()[0]["type"])

members: ['Ada', 'Alan'] | role: admin | tier: pro
bad tier -> literal_error


## 9. Custom validation with `field_validator`

Built-in constraints don't cover everything. A **`field_validator`** runs your own function on a
field's value. `mode="after"` runs *after* type coercion (you receive the typed value); return the
(possibly transformed) value, or `raise ValueError(...)` to reject it. Pydantic wraps your error
into a `ValidationError`.

In [10]:
from pydantic import field_validator

class Signup(BaseModel):
    email: str
    username: str

    @field_validator("email")
    @classmethod
    def must_look_like_email(cls, v: str) -> str:
        if "@" not in v or "." not in v.split("@")[-1]:
            raise ValueError("not a valid email address")
        return v.lower()               # normalize while we're here

    @field_validator("username")
    @classmethod
    def strip_and_check(cls, v: str) -> str:
        v = v.strip()
        if len(v) < 3:
            raise ValueError("username too short")
        return v

s = Signup(email="Ada@Example.COM", username="  ada  ")
print("normalized:", s.email, "|", repr(s.username))
try:
    Signup(email="nope", username="ok")
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])

normalized: ada@example.com | 'ada'
rejected: Value error, not a valid email address


## 10. Cross-field checks with `model_validator`

Some rules involve **several fields at once** (e.g. "end date must be after start date"). A
**`model_validator(mode="after")`** runs once the whole object is built, so you can compare fields.
Return `self`, or raise to reject.

In [11]:
from pydantic import model_validator

class DateRange(BaseModel):
    start_year: int
    end_year: int

    @model_validator(mode="after")
    def check_order(self):
        if self.end_year < self.start_year:
            raise ValueError("end_year must be >= start_year")
        return self

print("valid:", DateRange(start_year=2015, end_year=2020).model_dump())
try:
    DateRange(start_year=2020, end_year=2015)
except ValidationError as e:
    print("rejected:", e.errors()[0]["msg"])

valid: {'start_year': 2015, 'end_year': 2020}
rejected: Value error, end_year must be >= start_year


## 11. Computed fields (derived values)

A **`computed_field`** is a read-only property derived from other fields that is *also included in
serialization*. Use it when a value is always a function of the others (area from width×height),
so it can never get out of sync.

In [12]:
from pydantic import computed_field

class Rectangle(BaseModel):
    width: float
    height: float

    @computed_field
    @property
    def area(self) -> float:
        return self.width * self.height

r = Rectangle(width=3, height=4)
print("area:", r.area)
print("serialized (note 'area' is included):", r.model_dump())

area: 12.0
serialized (note 'area' is included): {'width': 3.0, 'height': 4.0, 'area': 12.0}


## 12. Serialization — objects back to dict / JSON

Turn a model back into plain data with **`model_dump()`** (→ dict) or **`model_dump_json()`**
(→ JSON string). Useful options: `exclude_none=True` (drop `None`s), `include`/`exclude` (pick
fields), `by_alias=True` (use the field aliases as keys).

In [13]:
acc = Account(username="ada", nickname=None, tags=["x"])
print("dict            :", acc.model_dump())
print("drop Nones      :", acc.model_dump(exclude_none=True))
print("only some fields:", acc.model_dump(include={"username", "tags"}))
print("json string     :", acc.model_dump_json())

dict            : {'username': 'ada', 'is_active': True, 'nickname': None, 'tags': ['x']}
drop Nones      : {'username': 'ada', 'is_active': True, 'tags': ['x']}
only some fields: {'username': 'ada', 'tags': ['x']}
json string     : {"username":"ada","is_active":true,"nickname":null,"tags":["x"]}


## 13. Parsing — untyped input into typed objects

The reverse direction. **`model_validate(data)`** builds a model from a Python dict/list;
**`model_validate_json(text)`** parses a JSON string directly. This is the workhorse for loading
files or API/LLM payloads into validated objects — with all the checks above applied.

In [14]:
raw_dict = {"name": "Grace", "age": "45"}          # note: age is a string here
person = Person.model_validate(raw_dict)           # validated + coerced
print("from dict:", person, "| age type:", type(person.age).__name__)

raw_json = '{"name": "Ada", "headquarters": {"city": "London", "country": "UK"}}'
company = Company.model_validate_json(raw_json)     # parse JSON -> nested objects
print("from json:", company.headquarters.city, company.headquarters.country)

from dict: name='Grace' age=45 | age type: int
from json: London UK


## 14. Aliases in practice (`populate_by_name`)

Incoming JSON often uses names you don't want as Python attributes (`publicationNumber` →
`pub_number`). Set an **`alias`** on the field. By default the model then expects the *alias* on
input; enabling **`populate_by_name=True`** lets you use *either* the field name or the alias.

In [15]:
from pydantic import ConfigDict

class Patent(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    pub_number: str = Field(alias="publicationNumber")

by_alias = Patent.model_validate({"publicationNumber": "US9081550B2"})   # JSON-style key
by_name = Patent(pub_number="US9081550B2")                              # python-style name
print("both work:", by_alias.pub_number, "==", by_name.pub_number)
print("dump with aliases:", by_alias.model_dump(by_alias=True))

both work: US9081550B2 == US9081550B2
dump with aliases: {'publicationNumber': 'US9081550B2'}


## 15. Model configuration (`ConfigDict`)

**`model_config = ConfigDict(...)`** changes how a model behaves. Common switches:

- **`extra="forbid"`** — reject unknown keys (catch typos) instead of silently ignoring them.
- **`frozen=True`** — make instances **immutable** (hashable, safe to share).
- **`str_strip_whitespace=True`** — trim strings automatically.

In [16]:
class Config(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True, str_strip_whitespace=True)
    host: str
    port: int

c = Config(host="  localhost  ", port=8080)
print("whitespace stripped:", repr(c.host))
try:
    c.port = 9090                                  # frozen -> cannot mutate
except ValidationError as e:
    print("frozen:", e.errors()[0]["type"])
try:
    Config(host="h", port=1, debug=True)           # unknown key 'debug'
except ValidationError as e:
    print("extra forbidden:", e.errors()[0]["type"])

whitespace stripped: 'localhost'
frozen: frozen_instance
extra forbidden: extra_forbidden


## 16. Rich standard types

Pydantic understands many stdlib types out of the box and coerces sensible inputs: `datetime`,
`date`, `UUID`, `Decimal`, `Path`, `HttpUrl`, and more. ISO-8601 strings become real `datetime`s,
etc.

In [17]:
from datetime import date, datetime
from uuid import UUID

class Event(BaseModel):
    id: UUID
    name: str
    starts: datetime
    day: date

e = Event(id="12345678-1234-5678-1234-567812345678", name="launch",
          starts="2026-08-28T09:30:00", day="2026-08-28")
print("id is a UUID   :", isinstance(e.id, UUID))
print("starts is dt   :", e.starts, type(e.starts).__name__)
print("day is a date  :", e.day, type(e.day).__name__)

id is a UUID   : True
starts is dt   : 2026-08-28 09:30:00 datetime
day is a date  : 2026-08-28 date


## 17. Settings from environment variables (`pydantic-settings`)

A sibling library, **`pydantic-settings`**, reads configuration from **environment variables**
(and `.env` files) into a typed model — the same validation, applied to config. This series uses
it to keep the optional live-LLM settings out of the code.

In [18]:
import os
from pydantic_settings import BaseSettings

class LLMSettings(BaseSettings):
    llm_model: str = "mock"       # reads env var LLM_MODEL if set, else this default
    llm_timeout: int = 30         # reads LLM_TIMEOUT, coerced to int

os.environ["LLM_MODEL"] = "deepseek-chat"
os.environ["LLM_TIMEOUT"] = "60"
settings = LLMSettings()
print("from environment:", settings.model_dump())
for k in ("LLM_MODEL", "LLM_TIMEOUT"):
    os.environ.pop(k, None)       # tidy up

from environment: {'llm_model': 'deepseek-chat', 'llm_timeout': 60}


## 18. Putting it together — the patent models of this series

Everything above is exactly what `patentrag/models.py` uses. `PatentClaim` has a constrained
field and a computed-style property; `Citation` builds a stable label; `PatentDocument` nests
lists of claims/sections and parses a real bundled patent from JSON with `from_corpus_json`
(a classmethod wrapper around the same validation you just learned).

In [19]:
from patentrag.models import PatentClaim, Citation, PatentDocument
import json

# a constrained, validated claim (number >= 1)
claim = PatentClaim(number=1, text="A method comprising ...", dependent_on=None)
print("independent?", claim.is_independent, "| dump:", claim.model_dump())

# a Citation and its stable label (used in prompts + verification)
cit = Citation(document_id="US9081550B2", publication_number="US9081550B2", claim_number=1, chunk_id="abc123")
print("citation label:", cit.label())

# parse a REAL bundled patent (nested claims + sections) via the same validation machinery
path = sorted((bs.DATA / "corpus").glob("US*.json"))[0]
doc = PatentDocument.from_corpus_json(json.loads(path.read_text(encoding="utf-8")))
print(f"\nparsed {doc.doc_id}: {len(doc.claims)} claims, {len(doc.sections)} sections, "
      f"{len(doc.independent_claims)} independent")

independent? True | dump: {'number': 1, 'text': 'A method comprising ...', 'dependent_on': None}
citation label: [PATENT=US9081550B2 | CLAIM=1 | CHUNK=abc123]

parsed US10083169B1: 15 claims, 6 sections, 3 independent


### You now know the whole toolkit

models · fields · validation & `ValidationError` · coercion (lax/strict) · optionals & defaults ·
`Field` constraints · nesting · collections/enums/`Literal` · `field_validator` ·
`model_validator` · `computed_field` · serialization (`model_dump`/`_json`) · parsing
(`model_validate`/`_json`) · aliases · `ConfigDict` · rich types · settings.

That is every Pydantic idea used anywhere in this series. Next appendices will do the same
zero-to-working treatment for other building blocks.

## Chapter invariants

In [20]:
# The concepts behave as taught (quick self-checks).
assert Person(name="Ada", age="36").age == 36                     # coercion
try:
    Person(name="Ada", age="x"); raised = False
except ValidationError:
    raised = True
assert raised                                                     # validation catches bad data
assert Rectangle(width=2, height=5).area == 10                    # computed field
assert Patent(pub_number="US1B2").pub_number == "US1B2"           # alias populate_by_name
assert doc.doc_id.startswith("US") and doc.claims                 # real model parses
print("All Appendix A1 invariants hold.")

All Appendix A1 invariants hold.


In [21]:
# === Chapter A1 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['pydantic', 'pydantic-settings']
print("Chapter A1 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER A1 VALIDATION: PASS")

Chapter A1 — environment
  Python : 3.12.10 on Windows 11
  pydantic                : 2.13.3
  pydantic-settings       : 2.14.0

CHAPTER A1 VALIDATION: PASS
